# Motor frequency-domain features with FFT and Goertzel

This notebook creates a **synthetic** motor-current signal, extracts FFT features, and evaluates the same targets with the Goertzel algorithm. Replace the simulated signal with measured vibration or current data once you know its sample rate and units.

Learning goals:
- connect sampling settings to the frequency axis
- extract peaks, band power and harmonic ratios
- understand when Goertzel is useful
- build one feature row per measurement window

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
fs = 12_000            # samples per second
duration = 1.0        # seconds
N = int(fs * duration)
t = np.arange(N) / fs

shaft_hz = 30.0
line_hz = 60.0
fault_hz = 240.0       # illustrative diagnostic band
print(f'N={N}, Nyquist={fs/2:.0f} Hz, bin spacing={fs/N:.2f} Hz')

## 1. Create a simulated measurement

The small 240 Hz component represents a condition-related feature. The notebook uses an amplitude change to create different examples later. Real fault behavior depends on the machine, sensor and operating point.

In [ ]:
def simulate_motor_signal(fault_amplitude=0.08, noise_std=0.10):
    fundamental = 1.0 * np.sin(2*np.pi*line_hz*t)
    rotor = 0.20 * np.sin(2*np.pi*shaft_hz*t + 0.5)
    harmonic = 0.12 * np.sin(2*np.pi*2*line_hz*t)
    fault = fault_amplitude * np.sin(2*np.pi*fault_hz*t)
    return fundamental + rotor + harmonic + fault + rng.normal(0, noise_std, N)

x = simulate_motor_signal()
plt.figure(figsize=(11, 3))
plt.plot(t[:1200], x[:1200], lw=0.8)
plt.xlabel('Time (s)'); plt.ylabel('Current (arbitrary units)')
plt.title('First 0.1 s of a simulated motor signal'); plt.grid(alpha=0.25)

## 2. Build a one-sided FFT spectrum

Remove the mean, apply a Hann window, and use `rfft` because the signal is real-valued. The amplitude scaling below corrects approximately for the window's coherent gain.

In [ ]:
def fft_spectrum(x, fs):
    x0 = x - np.mean(x)
    window = np.hanning(len(x0))
    X = np.fft.rfft(x0 * window)
    f = np.fft.rfftfreq(len(x0), d=1/fs)
    amplitude = 2 * np.abs(X) / window.sum()
    amplitude[0] /= 2
    return f, amplitude

f, amp = fft_spectrum(x, fs)
plt.figure(figsize=(11, 4))
plt.plot(f, amp, lw=1)
plt.xlim(0, 500); plt.ylim(0, 1.2)
plt.xlabel('Frequency (Hz)'); plt.ylabel('Approx. amplitude')
plt.title('One-sided FFT magnitude spectrum'); plt.grid(alpha=0.25)
for hz in [shaft_hz, line_hz, 2*line_hz, fault_hz]:
    plt.axvline(hz, color='tab:red', alpha=0.35, ls='--')

## 3. Turn the spectrum into features

A feature should answer a measurement question. Here we estimate a local peak, integrate band power, and compare a harmonic with the line-frequency fundamental.

In [ ]:
def peak_near(f, amplitude, center_hz, half_width_hz=2):
    mask = (f >= center_hz-half_width_hz) & (f <= center_hz+half_width_hz)
    return amplitude[mask].max()

def band_power(f, amplitude, lo_hz, hi_hz):
    mask = (f >= lo_hz) & (f <= hi_hz)
    return np.trapz(amplitude[mask]**2, f[mask])

def fft_features(x):
    f, a = fft_spectrum(x, fs)
    line = peak_near(f, a, line_hz)
    return {
        'shaft_1x_amp': peak_near(f, a, shaft_hz),
        'line_60_amp': line,
        'line_120_to_60_ratio': peak_near(f, a, 2*line_hz) / (line + 1e-12),
        'fault_240_amp': peak_near(f, a, fault_hz),
        'fault_band_power_220_260': band_power(f, a, 220, 260),
    }

pd.Series(fft_features(x)).round(4)

## 4. Compute selected bins with Goertzel

Goertzel evaluates a DFT bin through a recurrence. It works best when targets align with DFT bins. In this example, 1 Hz spacing means 60, 120 and 240 Hz align exactly.

In [ ]:
def goertzel_amplitude(x, fs, target_hz):
    # For cleanest interpretation choose target_hz on the DFT grid: k * fs / N
    x0 = x - np.mean(x)
    window = np.hanning(len(x0))
    k = int(round(target_hz * len(x0) / fs))
    omega = 2 * np.pi * k / len(x0)
    coeff = 2 * np.cos(omega)
    s_prev = 0.0
    s_prev2 = 0.0
    for sample in x0 * window:
        s = sample + coeff * s_prev - s_prev2
        s_prev2, s_prev = s_prev, s
    power = s_prev2**2 + s_prev**2 - coeff * s_prev * s_prev2
    return 2 * np.sqrt(power) / window.sum()

targets = [shaft_hz, line_hz, 2*line_hz, fault_hz]
comparison = pd.DataFrame({
    'frequency_hz': targets,
    'FFT amplitude': [peak_near(f, amp, h, 0.1) for h in targets],
    'Goertzel amplitude': [goertzel_amplitude(x, fs, h) for h in targets],
})
comparison.round(4)

## 5. Make a small feature table

Each row represents one window. The labels below are simulated, so treat this as a pipeline example rather than a fault model.

In [ ]:
rows = []
for label, fault_amp in [('healthy', 0.02), ('condition_A', 0.12)]:
    for record in range(30):
        features = fft_features(simulate_motor_signal(fault_amplitude=fault_amp))
        features['label'] = label
        features['record'] = record
        rows.append(features)
feature_table = pd.DataFrame(rows)
feature_table.groupby('label')['fault_240_amp'].agg(['mean', 'std', 'min', 'max']).round(4)

## 6. Before fitting a classifier

- Split by machine, run or time block, not random adjacent windows.
- Fit scaling and feature selection on the training split only.
- Test at operating points that differ from the training data.
- If speed changes, track speed, use wider bands, or convert frequency features to orders.
- Save the sample rate, window, preprocessing and feature definitions with the model.